# Experiment 5 — Missing Data Robustness

Simulates missing lab values at rates of 10%, 20%, and 40% (MCAR)
and measures AUROC degradation with and without MICE imputation.

**Key outputs:**
- AUROC vs missingness rate line chart
- MICE vs mean-imputation comparison
- Feature-wise missingness impact heatmap

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.seeding import seed_everything
from src.features.feature_pipeline import CancerTriageFeaturePipeline, load_horizon_data, create_train_val_test_split
from src.models.baselines import BaselineModelTrainer
from src.features.missing_data import missingness_robustness_experiment

seed_everything(42)
sns.set_theme(style='whitegrid', font_scale=1.2)
os.makedirs('../results/figures', exist_ok=True)
print('Ready.')

In [ ]:
cfg = {
    'seed': 42,
    'paths': {'data_processed': '../data/processed', 'figures': '../results/figures'},
    'features': {'cbc': ['wbc','hemoglobin','platelets','neutrophils','lymphocytes'],
                 'metabolic': ['glucose','albumin','creatinine'],
                 'inflammatory': ['crp']}
}

X, y = load_horizon_data('../data/processed', horizon_months=12, dataset='mimic')
X_train, X_val, X_test, y_train, y_val, y_test = create_train_val_test_split(X, y)

pipeline = CancerTriageFeaturePipeline(cfg)
X_train_p = pipeline.fit_transform(X_train, y_train)
X_val_p   = pipeline.transform(X_val)
X_test_p  = pipeline.transform(X_test)

trainer = BaselineModelTrainer(cfg)
results = trainer.train_all(X_train_p, y_train, X_val_p, y_val)
best = results['xgboost']['model']

print('Model trained. Running robustness experiment...')

In [ ]:
robustness_df = missingness_robustness_experiment(
    model=best,
    X_test=X_test_p,
    y_test=y_test,
    rates=[0.0, 0.10, 0.20, 0.40],
    n_trials=5
)
print(robustness_df.to_string())

robustness_df.to_csv('../results/exp5_missing_data_robustness.csv', index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
if 'missing_rate' in robustness_df.columns and 'auroc_mean' in robustness_df.columns:
    ax.plot(robustness_df['missing_rate'], robustness_df['auroc_mean'], 'o-', color='#2196F3', lw=2, markersize=8)
    if 'auroc_std' in robustness_df.columns:
        ax.fill_between(
            robustness_df['missing_rate'],
            robustness_df['auroc_mean'] - robustness_df['auroc_std'],
            robustness_df['auroc_mean'] + robustness_df['auroc_std'],
            alpha=0.3, color='#2196F3'
        )
ax.set(title='AUROC vs Missingness Rate (MCAR)', xlabel='Missing Rate', ylabel='AUROC')
ax.set_ylim(0.5, 1.0)
plt.tight_layout()
plt.savefig('../results/figures/exp5_missing_data.png', dpi=150, bbox_inches='tight')
plt.show()